In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "engelmann2012five")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Engelmann_2012_Data_Experiment1_Stealing.csv")
complete_path_2 = os.path.join(original_data_pathway, "Engelmann_2012_Data_Experiment2_Helping.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, 'stealing','1'],
                    [df2, 'helping','2']]
for x,y,k in experiment_import:
    x['experiment_name']=y
    x['experiment']=k

# df1.columns

In [3]:
data_frames=[df1, df2]


In [4]:
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"subject": "ape",
        "trial nr": "trial",
        "subgroup": "group_original"})
    x['study_id']="engelmann2012five"
    data_frames[index]=x
new_df=data_frames[0]


In [5]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [6]:

fulldf['ape'] = fulldf['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns
fulldf.rename(columns={"ape": "participant",
    "month ":"month"}, inplace=True)

In [7]:
age_calc_update = [['year_temp','year'],
                   ['month_temp','month'],
                   ['day_temp','day']]

for x,y in age_calc_update:
    fulldf[x]=fulldf[y]
    fulldf[x].replace( np.nan,0, inplace=True)
    fulldf[x]=fulldf[x].astype(int)




In [8]:
fulldf['condition'].replace('rc', 'refreshment_door_closed', inplace=True, regex=True)
fulldf['condition'].replace('ro', 'refreshment_door_open', inplace=True, regex=True)

In [9]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year_temp'].astype(str) + '-' + fulldf['month_temp'].astype(str) + '-' + fulldf['day_temp'].astype(str)
fulldf['dodc'].replace('0-0-0',np.nan, inplace=True)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [10]:
fulldf=fulldf[['study_id', 'experiment','experiment_name', 'year','month','day',  'participant', 'age_in_years',
       'sex', 'species',   'trial', 'condition',
       'behaviour', 'result']]

In [11]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'engelmann2012five_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'engelmann2012five_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)